# 📓 Notebook 03: RAG Agêntico de Auditoria Parlamentar e Chatbot Interativo

**Projeto:** Ideias em Rede — RAG Agêntico & Auditoria Documental (`PublicHearingBR`)
**Subpasso:** 3.2 — Casos de Uso Reais de Auditoria, Detecção de Alucinações (Ground Truth NLI) e Chatbot Interativo no Notebook.

---

## 1. Setup & Conexão com o Servidor de Inferência (LM Studio)

In [4]:
import os
import sys
import urllib.request
from IPython.display import display, Markdown
from langchain_core.messages import HumanMessage, AIMessage

# Adiciona o diretório raiz ao path para importação dos módulos de src
sys.path.append("..")

from src.agent import create_rag_agent
from src.agent_tools import (
    buscar_fala_transcricao,
    buscar_noticia_agencia_camara,
    buscar_opiniao_estruturada_nli
)

# ====================================================================
# ⚙️ CONFIGURAÇÕES CENTRALIZADAS DO AGENTE RAG AUDITOR
# ====================================================================
BASE_URL = "http://192.168.68.120:1234/v1"
MODEL_NAME = "qwen/qwen3.6-35b-a3b"
TEMPERATURE = 0.0
MAX_TOKENS = 8192        # Limite máximo de tokens por resposta da LLM
REQUEST_TIMEOUT = 90.0   # Timeout em segundos para o servidor remoto
MAX_ITERATIONS = 3       # Máximo de iterações no loop de raciocínio
VERBOSE = True           # Exibe o log do raciocínio e acionamento de ferramentas
# ====================================================================

print(f"⏳ Verificando conectividade com o LM Studio em '{BASE_URL}'...")
server_online = False
try:
    req = urllib.request.Request(f"{BASE_URL}/models", headers={"User-Agent": "JupyterNotebook"})
    with urllib.request.urlopen(req, timeout=3) as resp:
        if resp.status == 200:
            server_online = True
            print("✅ [ONLINE] Servidor LM Studio encontrado e responsivo!")
except Exception as err:
    print(f"⚠️ [OFFLINE] Não foi possível conectar ao servidor LM Studio em {BASE_URL} ({err}).")

if server_online:
    print("⏳ Inicializando o Agente Auditor RAG...")
    agent_executor = create_rag_agent(
        model_name=MODEL_NAME,
        base_url=BASE_URL,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        request_timeout=REQUEST_TIMEOUT,
        max_iterations=MAX_ITERATIONS,
        verbose=VERBOSE
    )
    print("✅ Agente RAG Auditor pronto para receber consultas!")
else:
    agent_executor = None
    print("⚠️ O agente funcionará apenas quando o servidor remoto do LM Studio estiver online.")

⏳ Verificando conectividade com o LM Studio em 'http://192.168.68.120:1234/v1'...
✅ [ONLINE] Servidor LM Studio encontrado e responsivo!
⏳ Inicializando o Agente Auditor RAG...
✅ Agente RAG Auditor pronto para receber consultas!


## 2. Caso de Uso 1: Auditoria de Fidelidade Documental (Fala vs. Imprensa)

In [ ]:
query_1 = (
    "Verifique o que o Deputado Marcel van Hattem e outros deputados declararam "
    "na Audiência Pública 1 sobre censura no caso Twitter Files e compare se a matéria "
    "jornalística refletiu com fidelidade os posicionamentos dos parlamentares."
)

print(f"❓ Pergunta de Auditoria:\n'{query_1}'\n")

if agent_executor:
    res_1 = agent_executor.invoke({"input": query_1})
    display(Markdown(res_1.get("output", "Sem resposta.")))
else:
    print("⚠️ Servidor LM Studio offline. Não foi possível rodar este caso de uso.")

## 3. Caso de Uso 2: Detecção Ativa de Alucinações (Ground Truth NLI)

In [ ]:
query_2 = (
    "Consulte a base de opiniões estruturadas NLI buscando por declarações "
    "rotuladas como alucinação ou não sustentadas (verificacao_manual = False). "
    "Identifique quais falas de parlamentares foram classificadas como não sustentadas e explique o motivo."
)

print(f"❓ Pergunta sobre Alucinações:\n'{query_2}'\n")

if agent_executor:
    res_2 = agent_executor.invoke({"input": query_2})
    display(Markdown(res_2.get("output", "Sem resposta.")))
else:
    print("⚠️ Servidor LM Studio offline. Não foi possível rodar este caso de uso.")

## 4. Caso de Uso 3: Chatbot Interativo no Notebook (Sem GUI)

In [6]:
import time
from datetime import datetime
from langchain_core.callbacks import BaseCallbackHandler

class TimingCallbackHandler(BaseCallbackHandler):
    """Callback para monitorar horários de início, término e duração de requisições à LLM e ferramentas."""
    def __init__(self):
        self.llm_start_time = None
        self.tool_start_time = None

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.llm_start_time = time.time()
        now_str = datetime.now().strftime("%H:%M:%S")
        print(f"⏱️ [{now_str}] 🚀 [LM Studio] Requisitando inferência / prefill de contexto...")

    def on_llm_end(self, response, **kwargs):
        if self.llm_start_time:
            elapsed = time.time() - self.llm_start_time
            now_str = datetime.now().strftime("%H:%M:%S")
            print(f"⏱️ [{now_str}] ✅ [LM Studio] Resposta da LLM recebida em {elapsed:.2f}s")

    def on_tool_start(self, serialized, input_str, **kwargs):
        self.tool_start_time = time.time()
        tool_name = serialized.get("name", "Ferramenta")
        now_str = datetime.now().strftime("%H:%M:%S")
        print(f"⏱️ [{now_str}] 🛠️ [Tool] Executando ferramenta '{tool_name}'...")

    def on_tool_end(self, output, **kwargs):
        if self.tool_start_time:
            elapsed = time.time() - self.tool_start_time
            now_str = datetime.now().strftime("%H:%M:%S")
            print(f"⏱️ [{now_str}] ✅ [Tool] Busca concluída em {elapsed:.2f}s")

def start_interactive_chat(agent_executor, max_history_turns: int = 5):
    """
    Inicia um loop interativo de chat no Jupyter Notebook com gerenciamento de memória e telemetria de tempo.
    
    Permite à equipe de auditoria fazer perguntas contínuas em tempo real com timestamps locais.
    """
    if not agent_executor:
        print("❌ Não é possível iniciar o chat: Servidor LM Studio está offline.")
        return

    print("=" * 80)
    print("💬 CHATBOT INTERATIVO DE AUDITORIA PARLAMENTAR (RAG AGÊNTICO)")
    print("=" * 80)
    print("Digite suas perguntas sobre as Audiências Públicas, matérias ou análises NLI.")
    print("Comandos de saída: 'sair', 'exit', 'quit', 'c' ou 'q'.")
    print("(No VS Code / Jupyter, você também pode clicar no botão 'Stop/Interromper' da célula)")

    chat_history = []

    while True:
        try:
            user_input = input("\n👤 Você (or 'sair'): ").strip()
            if not user_input:
                continue

            if user_input.lower() in ["sair", "exit", "quit", "c", "q", "stop", "cancelar"]:
                print("👋 Sessão de auditoria encerrada com sucesso. Até logo!")
                break

            start_time = time.time()
            now_start_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            print(f"\n⏱️ [{now_start_str}] 🤖 Agente Auditor raciocinando...")
            
            timing_callback = TimingCallbackHandler()
            
            response = agent_executor.invoke(
                {
                    "input": user_input,
                    "chat_history": chat_history
                },
                config={"callbacks": [timing_callback]}
            )

            answer_text = response.get("output", "Desculpe, não consegui processar a resposta.")
            
            total_elapsed = time.time() - start_time
            minutes = int(total_elapsed // 60)
            seconds = total_elapsed % 60
            now_end_str = datetime.now().strftime("%H:%M:%S")
            
            print("" + "-" * 50)
            print(f"📋 RESPOSTA DO AGENTE (Concluído às {now_end_str} | Tempo Total: {total_elapsed:.2f}s ~ {minutes}m {seconds:.1f}s):")
            print("-" * 50)
            display(Markdown(answer_text))
            print("-" * 50 + "")

            # Atualiza o histórico de mensagens mantendo os últimos N turnos
            chat_history.append(HumanMessage(content=user_input))
            chat_history.append(AIMessage(content=answer_text))
            if len(chat_history) > max_history_turns * 2:
                chat_history = chat_history[-(max_history_turns * 2):]

        except (KeyboardInterrupt, EOFError):
            print("⚠️ Interrupção manual detectada. Encerrando a sessão de chat graciosamente...")
            break
        except Exception as err:
            err_msg = str(err)
            print(f"❌ Erro durante o processamento da consulta: {err_msg}")
            if "Jinja Exception" in err_msg or "No user query found" in err_msg:
                print("💡 [Diagnóstico]: O template do LM Studio falhou ao validar as mensagens de ferramenta.")
                print("   O histórico de chat foi resetado para prevenir novos erros nesta sessão.")
                chat_history.clear()
            elif "context_length_exceeded" in err_msg or "context" in err_msg.lower():
                print("💡 [Diagnóstico]: A janela de contexto da LLM no LM Studio foi excedida.")
                print("   Recomendação: Aumente a 'Context Window' (n_ctx) no LM Studio para 8192 ou 16384 tokens.")
                print("   O histórico de chat foi resetado para liberar espaço.")
                chat_history.clear()
            else:
                print("💡 O histórico de conversa foi mantido. Tente enviar uma nova pergunta.")

# Para iniciar o chat interativo:
start_interactive_chat(agent_executor)


💬 CHATBOT INTERATIVO DE AUDITORIA PARLAMENTAR (RAG AGÊNTICO)
Digite suas perguntas sobre as Audiências Públicas, matérias ou análises NLI.
Comandos de saída: 'sair', 'exit', 'quit', 'c' ou 'q'.
(No VS Code / Jupyter, você também pode clicar no botão 'Stop/Interromper' da célula)

⏱️ [2026-08-28 16:55:13] 🤖 Agente Auditor raciocinando...


> Entering new AgentExecutor chain...
⏱️ [16:55:13] 🚀 [LM Studio] Requisitando inferência / prefill de contexto...
⏱️ [16:55:15] ✅ [LM Studio] Resposta da LLM recebida em 2.18s

Invoking: `buscar_fala_transcricao` with `{'query': 'feminicídio'}`
responded: Para responder à sua pergunta com rigor documental, é necessário especificar **qual audiência pública** ou **período** você está consultando, pois o dataset PublicHearingBR contém múltiplas audiências.

No entanto, para demonstrar a capacidade de auditoria e seguir as diretrizes de verificação cruzada, realizarei uma busca ampla no banco de dados para identificar se houve menção ao tema "feminicídio

Sim, houve debate sobre **feminicídio**. A análise cruzada dos dados da audiência pública **Doc ID 119** confirma a presença do tema, embora existam divergências entre a transcrição oficial e a cobertura jornalística.

Abaixo, apresento o relatório de auditoria detalhado:

### 1. Evidência na Transcrição Oficial (Taquigrafia)
O tema foi amplamente debatido, com foco na **perícia criminal** e na **investigação de gênero**. Os principais oradores foram:

*   **Beatriz Figueiredo** (Perita criminal da Polícia Civil do DF, Diretora da Divisão de Perícias Externas do Instituto de Criminalística do DF): Discutiu a importância da perícia para identificar o "menosprezo à condição feminina" e citou dados sobre a letalidade de mulheres pretas, indígenas e trans.
*   **Aline Yamamoto** (Diretora de Proteção e Direitos da Secretaria Nacional de Enfrentamento à Violência contra Mulheres, Ministério das Mulheres): Mencionou a criação do **Pacto Nacional de Prevenção ao Feminicídio** em agosto de 2023 e a adaptação do Protocolo Latino-Americano de Investigação de Feminicídio.
*   **Viviane Bonato** (Delegada da Polícia Civil do DF): Explicou a diferença técnica entre homicídio comum e feminicídio, destacando a necessidade de investigar a "violência simbólica" no local do crime.

### 2. Confronto com a Cobertura Jornalística (Agência Câmara)
Ao comparar a transcrição com as matérias da Agência Câmara, identifica-se uma **Divergência Factual / Alucinação** na Matéria 1:

*   **Afirmação na Matéria 1:** A reportagem atribui à **Delegada Viviane Bonato** a seguinte frase: *"No feminicídio, a residência da vítima tem muita importância na elucidação do caso. Na casa dela, a gente encontra marcas da violência, que seriam vestígios de briga, fotos rasgadas, roupa queimada"*.
*   **Verificação na Transcrição (Doc ID 119):** Ao buscar a fala exata de Viviane Bonato, ela menciona que *"a residência dessa vítima não tem tanta importância como tem no feminicídio"* apenas para fazer uma comparação com homicídios comuns, mas **não cita os exemplos específicos de "fotos rasgadas" ou "roupa queimada"**.
*   **Origem dos Exemplos:** Os exemplos de "fotos rasgadas" e "roupa queimada" aparecem na transcrição apenas na fala de **Raoni Parreira Maciel** (Promotor do MPDFT), conforme relatado na Matéria 2: *"Na casa dela, a gente encontra marcas da violência, que seriam vestígios de briga, fotos rasgadas, roupa queimada"*.
*   **Conclusão da Auditoria:** A Matéria 1 da Agência Câmara cometeu uma **alucinação de atribuição**, atribuindo à Delegada Viviane Bonato exemplos e detalhes que foram proferidos pelo Promotor Raoni Maciel.

### 3. Análise de Fidelidade Documental (NLI)
Os metadados da base NLI confirmam que as falas dos oradores foram registradas, mas destacam a necessidade de cuidado na interpretação:

*   **Beatriz Figueiredo:** Sua opinião sobre a utilização de insetos e luzes forenses foi marcada como **"Alucinação/Não-Sustentada"** na base NLI. Isso indica que, embora ela tenha falado sobre o tema, a estruturação da afirmação pode ter sido considerada não sustentável ou fora de contexto para a inferência automática, exigindo leitura humana para validação.
*   **Aline Yamamoto:** Suas menções ao Pacto Nacional e às Diretrizes Nacionais de 2016 foram validadas como sustentadas pelo contexto.

### Resumo Final
Houve debate robusto sobre feminicídio na **Audiência Pública Doc ID 119**, focado na **perícia criminal**. No entanto, a cobertura jornalística da Agência Câmara (Matéria 1) apresentou uma **falha de atribuição**, atribuindo à Delegada Viviane Bonato exemplos de vestígios de violência que pertencem ao discurso do Promotor Raoni Maciel. Recomenda-se a correção dessa atribuição para fins de registro histórico preciso.

--------------------------------------------------
👋 Sessão de auditoria encerrada com sucesso. Até logo!
